## Export to GGUF Format

Convert the fine-tuned model to GGUF format for use with llama.cpp, Ollama, LM Studio, and other inference engines.

In [1]:
!pip install GitPython


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import requests
import pickle
import git
import os
import subprocess
import sys
from transformers import AutoTokenizer, AutoModelForCausalLM
import requests

In [3]:
SERIALIZED_MODEL_URL="http://localhost:50000/buckets/mlpipeline/v2/artifacts/finetuning-fnmodel/37c9bcc3-df34-4609-ac3e-4f9f8c8a5ccd/train-model-step/63737681-31c6-4379-bbb8-482fd83c0e8f/result_model_output_artifact"
BASE_MODEL_ID="Qwen/Qwen2.5-0.5B"

In [4]:
target_dir = "/tmp/llama.cpp"
repo_url = "https://github.com/ggerganov/llama.cpp.git"

# 1. Clone repository using GitPython if it doesn't already exist
if not os.path.exists(target_dir):
    print("Cloning llama.cpp...")
    git.Repo.clone_from(repo_url, target_dir, depth=1)
else:
    print("llama.cpp already cloned")

# 2. Install requirements using pip
requirements_file = os.path.join(target_dir, "requirements.txt")
if os.path.exists(requirements_file):
    print("Installing requirements...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", requirements_file],
        check=True
    )

llama.cpp already cloned
Installing requirements...


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
proto-plus 1.28.3 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
kfp-pipeline-spec 2.17.0 requires protobuf<7.0,>=6.31.1, but you have protobuf 4.25.9 which is incompatible.
googleapis-common-protos 1.75.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-api-core 2.34.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 4.25.9 which is incompatible.
google-api-core 2.34.0 requires requests<3.0.0,>=2.33.0, but you have requests 2.32.5 which is incompatible.
kfp 2.17.0 requires protobuf<7.0,>=6.31.1, but you have protobuf 4.25.9 which is incompatible.
kfp 2.17.0 requires requests==2.33.0; python_version >= "3.10", but you have requests 2.32.5 which is incompatible.
kfp-kubernetes 2.17.0 requires protobuf<7.0,>=6.33.5, but you hav

In [5]:
MODEL_DIR="./result_model"
gguf_output_dir = "./gguf"
os.makedirs(gguf_output_dir, exist_ok=True)

serialized_model = None

serialized_model_file = "/tmp/model.pkl"

if not serialized_model and SERIALIZED_MODEL_URL:
    print(f"Downloading model from {SERIALIZED_MODEL_URL}")
    with requests.get(SERIALIZED_MODEL_URL, stream=True) as r:
        r.raise_for_status()
        with open(serialized_model_file, 'wb') as f:
            for chunk in r.iter_content(chunk_size=8142): 
                f.write(chunk)
print(f"Download successful!")


Download successful!


In [11]:
print("Serializing and saving local model...")
with open(serialized_model_file, 'rb') as file:
    serialized_model = pickle.load(file)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID)

tokenizer.save_pretrained(MODEL_DIR)
serialized_model.generation_config.early_stopping = False
serialized_model.generation_config.num_return_sequences = 0
serialized_model.generation_config.num_beams = 0
serialized_model.save_pretrained(MODEL_DIR, strict=False)

print("Model serialized and stored locally sucessfully!")


Serializing and saving local model...
Model serialized and stored locally sucessfully!


In [ ]:
gguf_f16_path = f"{gguf_output_dir}/model-f16.gguf"

print(f"Converting to GGUF format...")
print(f"Output directory: {gguf_output_dir}")
convert_cmd = [
    "python", "/tmp/llama.cpp/convert_hf_to_gguf.py",
    MODEL_DIR,
    "--outfile", gguf_f16_path,
    "--outtype", "f16"
]

print(f"Running: {' '.join(convert_cmd)}")
result = subprocess.run(convert_cmd, capture_output=True, text=True)

if result.returncode == 0:
    print(f"F16 GGUF created: {gguf_f16_path}")
else:
    print(f"Conversion failed: {result.stderr}")
    raise RuntimeError("GGUF conversion failed")

print("\n" + "="*60)
print("Conversion to GGUF finished")
print("="*60)
print(f"GGUF F16 model:     {gguf_f16_path}")


Converting to GGUF format...
Output directory: ./gguf
Running: python /tmp/llama.cpp/convert_hf_to_gguf.py ./result_model --outfile ./gguf/model-f16.gguf --outtype f16
